# Module 6 — Decision trees and ensemble learning (credit scoring): commented walkthrough

Alexey's lecture code (lessons 6.1 – 6.9), with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the machine-learning logic behind it (what could go wrong if we skipped it)
- `# PYTHON` how the Python / NumPy / pandas / scikit-learn / XGBoost syntax works, so you can write it yourself later

The original, uncommented notebook is untouched in `../notebooks/`.

Runs end to end on pandas 3, scikit-learn 1.9, NumPy 2 and XGBoost 3.4 (checked). The forest loops in 6.6 train ~180 random forests: give that section a minute or two.

Differences from the lecture:
- the data is read from the local file `CreditScoring.csv` (same file the lecture downloads); `!wget`, `!head` and `!pip install xgboost` are dropped. Install XGBoost once in your environment (`pip install xgboost`, or `conda install -c conda-forge xgboost`);
- **XGBoost 3 refuses a NumPy array as `feature_names`**: the lecture's 6.9 cell `xgb.DMatrix(..., feature_names=dv.get_feature_names_out())` stops with *"Expecting a sequence of strings for feature names"*, and every cell after it fails or scores a stale model. Here it is wrapped in `list(...)`, as the lecture already does in 6.7;
- 6.3: the hand-written rule checks `client['job'] == 'parttime'`, but the data decodes job 2 as `'partime'` (one t), so that branch could never fire. Fixed to `'partime'`;
- 6.4: the lecture's single-threshold cell prints the **left** side's class shares twice; the second print now shows the right side. The misclassification table the lesson types by hand is computed in code. The lesson's `AVG` column is the plain mean of the two sides, although the text says "weighted by group sizes": the cell shows both, and `assets > 3000` wins either way;
- 6.5 / 6.6: the lecture picks the hyperparameters by eye from tables and plots. The picks are kept (they feed 6.9), and a cell after each loop shows the ranked numbers, so you can check the pick against your own run;
- 6.6: `n_jobs=-1` added to every `RandomForestClassifier` (uses all CPU cores, same trees and same AUC as without it);
- 6.7: the `%%capture output` + text-parsing route is kept once, because the lesson teaches it. Then the same curve is read from XGBoost's `evals_result=` dictionary, which is what the rest of the notebook uses: it does not break if XGBoost prints an extra warning line;
- 6.8: the lecture notebook keeps only the **last** run of each comparison (the other values were run in the video and overwritten), so its comparison plots show one curve, or none after `del scores['max_depth=10']`. Here a small helper trains every value the lesson compares: `eta` 0.3, 1.0, 0.1, 0.05, 0.01; `max_depth` 6, 3, 4, 10; `min_child_weight` 1, 10, 30;
- 6.9: the final `DictVectorizer` gets `.fillna(0)`, like every model before it. The lecture skips it there, so the final model is trained on a slightly different pipeline (NaN instead of 0) from the one it validated;
- no pandas 3 breakage in this lecture: the `map`, `replace`, `pivot` and `== 'default'` calls behave the same.

**HW6 uses a different dataset (car fuel efficiency, a regression task) and pins its own parameters**, so the numbers here do not carry over.

## 6.1 Credit risk scoring project

A bank decides whether to give a loan. From the application (income, assets, debt, job, ...) we predict the
probability that the client **defaults** (does not pay back). Target: `default` = 1, `ok` = 0. Binary
classification again, scored with ROC AUC from Module 4. New this week: the model family. Instead of
logistic regression's weighted sum, **trees** ask a sequence of yes/no questions.

In [ ]:
# PYTHON: 'import X as Y' loads a library under a short nickname.
import pandas as pd
import numpy as np

import seaborn as sns                 # heatmap of the tuning grid in 6.5
from matplotlib import pyplot as plt  # the same as 'import matplotlib.pyplot as plt'

# PYTHON: display() shows a DataFrame as a table from inside a loop (a bare expression only shows at the cell's end).
from IPython.display import display

%matplotlib inline

## 6.2 Data cleaning and preparation

- reading the dataset
- re-encoding the categorical variables (numbers → readable strings)
- missing values hidden as `99999999`
- train / validation / test split

In [ ]:
# GOAL: load the credit scoring data (4,455 loan applications, 14 columns).
df = pd.read_csv('CreditScoring.csv')

# PYTHON: .str.lower() lower-cases every column name at once ('Status' -> 'status').
df.columns = df.columns.str.lower()
df.head()

In [ ]:
# GOAL: look at the target. 1 = ok, 2 = default, 0 = one record with unknown status.
df.status.value_counts()

In [ ]:
# GOAL: replace the number codes by words, so the categories read as categories.
# WHY: DictVectorizer one-hot encodes STRING values and passes NUMBERS through as one numeric column.
#      Left as 1/2/3, 'home' would be treated as a quantity ("owner is twice rent"), which means nothing.
# PYTHON: Series.map(dict) looks every value up in the dictionary: 1 -> 'ok', 2 -> 'default', ...
status_values = {
    1: 'ok',
    2: 'default',
    0: 'unk'
}

df.status = df.status.map(status_values)

In [ ]:
# GOAL: the same decoding for the other four categorical columns (codes from the dataset's documentation).
home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

df.home = df.home.map(home_values)

marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

df.marital = df.marital.map(marital_values)

records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

df.records = df.records.map(records_values)

# NOTE: the dataset spells part-time as 'partime' (one t). Keep that spelling in mind for 6.3.
job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df.job = df.job.map(job_values)

df.head()

In [ ]:
# GOAL: scan the numerical columns. Look at the max of income, assets and debt.
# PYTHON: describe() gives count / mean / std / min / quartiles / max per numerical column; round() for readability.
df.describe().round()

In [ ]:
# GOAL: 99999999 is not a real income: it is the file's code for "missing". Turn it into NaN.
# WHY: left in, a handful of 99,999,999 values would dominate means and look like extreme clients to the model.
# PYTHON: replace(to_replace=a, value=b) swaps every a for b; np.nan is "not a number" = missing.
for c in ['income', 'assets', 'debt']:
    df[c] = df[c].replace(to_replace=99999999, value=np.nan)

# Now the max values are plausible and 'count' shows how many values are missing per column.
df.describe().round()

In [ ]:
# GOAL: drop the single record with unknown status.
# WHY: one row cannot teach a third class, and it is neither 'ok' nor 'default' for our 0/1 target.
# PYTHON: df[mask] keeps the rows where the mask is True; reset_index(drop=True) renumbers 0, 1, 2, ...
df = df[df.status != 'unk'].reset_index(drop=True)

In [ ]:
# GOAL: 60 / 20 / 20 split. The lecture uses random_state=11 (HW6 pins random_state=1 on its own data).
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=11)
# WHY: 0.25 of the remaining 80% = 20% of the whole.
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=11)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
# GOAL: the target as 0/1: 1 = default (the event we want to catch).
# PYTHON: (column == 'default') is True/False per row; .astype('int') -> 1/0; .values -> NumPy array.
y_train = (df_train.status == 'default').astype('int').values
y_val = (df_val.status == 'default').astype('int').values
y_test = (df_test.status == 'default').astype('int').values

In [ ]:
# WHY: remove the target from the inputs, or the model reads the answer.
#      (df_full_train keeps 'status' on purpose: the final model in 6.9 needs it.)
del df_train['status']
del df_val['status']
del df_test['status']

df_train

## 6.3 Decision trees

- what a decision tree looks like
- training a decision tree
- overfitting
- controlling the size of a tree

In [ ]:
# GOAL: a decision tree written by hand: nested if / else on the client's features.
# WHY: this is exactly what a trained tree is. Learning a tree = learning these conditions and thresholds from data.
# NOTE: the lecture tests 'parttime'; the data says 'partime' (see 6.2), so that branch could never fire.
def assess_risk(client):
    if client['records'] == 'yes':
        if client['job'] == 'partime':
            return 'default'
        else:
            return 'ok'
    else:
        if client['assets'] > 6000:
            return 'ok'
        else:
            return 'default'

In [ ]:
# PYTHON: .iloc[0] = first row (by position); .to_dict() -> {column: value} for that one client.
xi = df_train.iloc[0].to_dict()
assess_risk(xi)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text

In [ ]:
# GOAL: features -> matrix, as in Modules 3 and 4.
# WHY: scikit-learn trees do not accept NaN in this setup. fillna(0) says "no income / assets / debt reported".
#      For assets and debt that is a reasonable reading; for income it is a simplification the lecture accepts.
# PYTHON: to_dict(orient='records') -> one {column: value} dict per row, DictVectorizer's input format.
train_dicts = df_train.fillna(0).to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

In [ ]:
# GOAL: train a tree with default settings: no depth limit.
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

In [ ]:
# WHY: the validation set is transformed with the dv fitted on train (transform, never fit).
val_dicts = df_val.fillna(0).to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [ ]:
# GOAL: validation AUC of the unlimited tree.
# PYTHON: predict_proba(X)[:, 1] = probability of class 1 (default). A tree's probability = the share of
#         defaulters in the leaf the client lands in.
y_pred = dt.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, y_pred)

In [ ]:
# GOAL: the same tree scored on the data it was trained on.
# WHY: train AUC 1.0 vs validation ~0.65 = OVERFITTING. With no depth limit the tree keeps splitting until
#      every leaf is pure, so it memorises each training client and the rules do not generalise.
y_pred = dt.predict_proba(X_train)[:, 1]
roc_auc_score(y_train, y_pred)

In [ ]:
# GOAL: control the size of the tree: at most 2 levels of questions.
dt = DecisionTreeClassifier(max_depth=2)
dt.fit(X_train, y_train)

In [ ]:
# WHY: train and validation AUC are now close: the small tree learns general rules, not individual clients.
#      Too small is also bad (underfitting): 6.5 searches the depth in between.
y_pred = dt.predict_proba(X_train)[:, 1]
auc = roc_auc_score(y_train, y_pred)
print('train:', auc)

y_pred = dt.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)
print('val:', auc)

In [ ]:
# GOAL: look inside the tree: the learned rules as text.
# PYTHON: export_text needs the column names; dv.get_feature_names_out() gives them in matrix order.
#         One-hot columns read 'records=no <= 0.50', i.e. "records is NOT 'no'".
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

## 6.4 Decision tree learning algorithm

- finding the best split for one column
- finding the best split for the entire dataset
- stopping criteria
- decision tree learning algorithm

A toy dataset of eight clients, so every step is visible.

In [ ]:
# GOAL: one feature (assets) and the target (status).
data = [
    [8000, 'default'],
    [2000, 'default'],
    [   0, 'default'],
    [5000, 'ok'],
    [5000, 'ok'],
    [4000, 'ok'],
    [9000, 'ok'],
    [3000, 'default'],
]

df_example = pd.DataFrame(data, columns=['assets', 'status'])
df_example

In [ ]:
# WHY: sorted, the candidate thresholds are easy to see: a split can only fall between two observed values.
df_example.sort_values('assets')

In [ ]:
# GOAL: the candidate thresholds T for the condition 'assets > T' (each observed value).
Ts = [0, 2000, 3000, 4000, 5000, 8000]

In [ ]:
# GOAL: one split, T = 4000. Left = condition false (assets <= T), right = condition true (assets > T).
T = 4000
df_left = df_example[df_example.assets <= T]
df_right = df_example[df_example.assets > T]

# PYTHON: value_counts(normalize=True) gives class SHARES instead of counts.
display(df_left)
print(df_left.status.value_counts(normalize=True))
display(df_right)
# The lecture prints df_left here a second time; the right side is what we need.
print(df_right.status.value_counts(normalize=True))

In [ ]:
# GOAL: all thresholds, same printout (as in the lecture).
for T in Ts:
    print(T)
    df_left = df_example[df_example.assets <= T]
    df_right = df_example[df_example.assets > T]

    display(df_left)
    print(df_left.status.value_counts(normalize=True))
    display(df_right)
    print(df_right.status.value_counts(normalize=True))

    print()

In [ ]:
# GOAL: turn the printouts into the lesson's table: majority decision, impurity per side, average.
# WHY: each side predicts its MAJORITY class; the MISCLASSIFICATION RATE = share of the other class(es)
#      = how impure the group is. A pure group (one class only) has rate 0. Best split = lowest impurity.
def misclassification(df_side):
    # PYTHON: .max() of the class shares = the majority share; 1 - that = the share we get wrong.
    return 1 - df_side.status.value_counts(normalize=True).max()


def split_table(df, feature, Ts):
    rows = []
    for T in Ts:
        df_left = df[df[feature] <= T]
        df_right = df[df[feature] > T]

        imp_left = misclassification(df_left)
        imp_right = misclassification(df_right)

        # The lesson's AVG: plain mean of the two sides.
        avg = (imp_left + imp_right) / 2
        # Weighted by group size (what scikit-learn does): a big impure side counts more than a tiny pure one.
        weighted = (len(df_left) * imp_left + len(df_right) * imp_right) / len(df)

        # PYTHON: .mode()[0] = the most frequent value (the majority decision). On a 50/50 tie (T=5000)
        #         mode() lists both classes alphabetically, so [0] says 'default'; the impurity is 0.5 either way.
        rows.append((feature, T, df_left.status.mode()[0], imp_left,
                     df_right.status.mode()[0], imp_right, avg, weighted))

    columns = ['feature', 'T', 'decision_left', 'impurity_left',
               'decision_right', 'impurity_right', 'avg', 'weighted']
    return pd.DataFrame(rows, columns=columns)


split_table(df_example, 'assets', Ts).round(2)

`assets > 3000` has the lowest impurity with both averages (the left side is pure: everyone with assets ≤ 3000 defaulted).

Now two features: the algorithm tries every threshold of **every** feature and keeps the best condition overall.

In [ ]:
data = [
    [8000, 3000, 'default'],
    [2000, 1000, 'default'],
    [   0, 1000, 'default'],
    [5000, 1000, 'ok'],
    [5000, 1000, 'ok'],
    [4000, 1000, 'ok'],
    [9000,  500, 'ok'],
    [3000, 2000, 'default'],
]

df_example = pd.DataFrame(data, columns=['assets', 'debt', 'status'])
df_example

In [ ]:
df_example.sort_values('debt')

In [ ]:
# PYTHON: a dict maps each feature name to its list of candidate thresholds.
thresholds = {
    'assets': [0, 2000, 3000, 4000, 5000, 8000],
    'debt': [500, 1000, 2000]
}

In [ ]:
# GOAL: the lecture's double loop: every feature, every threshold.
# PYTHON: dict.items() yields (key, value) pairs: here (feature, list of thresholds).
for feature, Ts in thresholds.items():
    print('#####################')
    print(feature)
    for T in Ts:
        print(T)
        df_left = df_example[df_example[feature] <= T]
        df_right = df_example[df_example[feature] > T]

        display(df_left)
        print(df_left.status.value_counts(normalize=True))
        display(df_right)
        print(df_right.status.value_counts(normalize=True))

        print()
    print('#####################')

In [ ]:
# GOAL: the same search as one table, best condition first.
# WHY: this IS the split-finding step: FOR each feature, FOR each threshold, compute impurity, keep the lowest.
#      A real tree then repeats it on the left and right groups until a stopping rule fires:
#      group pure, max_depth reached, group smaller than min_samples_leaf, or max number of leaves reached.
# PYTHON: pd.concat stacks DataFrames vertically; sort_values orders by a column.
df_splits = pd.concat([split_table(df_example, f, Ts) for f, Ts in thresholds.items()])
df_splits.sort_values('weighted').round(2)

scikit-learn uses **Gini impurity** (default) or **entropy** instead of the misclassification rate (they react
more smoothly to small improvements), and **MSE** for regression trees. Same idea: the split with the lowest impurity wins.

## 6.5 Decision trees parameter tuning

- selecting `max_depth`
- selecting `min_samples_leaf`

In [ ]:
# GOAL: validation AUC for a range of depths. None = no limit.
# PYTHON: '%4s -> %.3f' % (depth, auc): depth as a 4-wide string (so None fits), AUC with 3 decimals.
depths = [1, 2, 3, 4, 5, 6, 10, 15, 20, None]

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth)
    dt.fit(X_train, y_train)

    y_pred = dt.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)

    print('%4s -> %.3f' % (depth, auc))

AUC rises to a peak around depth 4 – 6 and falls after it (overfitting). Deeper than ~10 the numbers jitter
between runs: `DecisionTreeClassifier` without `random_state` breaks ties between equal splits at random.

In [ ]:
# GOAL: grid search over the two size controls, keeping depth in the good range 4 - 6.
# WHY: min_samples_leaf = the smallest group a leaf may hold. Bigger leaves = smoother, less overfit probabilities.
# PYTHON: a nested loop tries every (depth, s) pair; each result is a tuple appended to a list.
scores = []

for depth in [4, 5, 6]:
    for s in [1, 5, 10, 15, 20, 500, 100, 200]:
        dt = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=s)
        dt.fit(X_train, y_train)

        y_pred = dt.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_pred)

        scores.append((depth, s, auc))

In [ ]:
columns = ['max_depth', 'min_samples_leaf', 'auc']
df_scores = pd.DataFrame(scores, columns=columns)

In [ ]:
# GOAL: reshape the long table into a grid: rows = min_samples_leaf, columns = max_depth.
# PYTHON: pivot(index=..., columns=..., values=...) turns the values of one column into column headers.
#         Rows come out sorted, so the lecture's unsorted list (... 500, 100, 200) ends up in order.
df_scores_pivot = df_scores.pivot(index='min_samples_leaf', columns=['max_depth'], values=['auc'])
df_scores_pivot.round(3)

In [ ]:
# GOAL: the same grid as a heatmap: the brightest cell is the best pair.
# PYTHON: annot=True writes the value in each cell; fmt='.3f' formats it with 3 decimals.
sns.heatmap(df_scores_pivot, annot=True, fmt=".3f")

In [ ]:
# GOAL: check the pick against your run: the top five pairs.
# WHY: the lecture picks max_depth=6, min_samples_leaf=15. Neighbouring cells differ in the 3rd decimal,
#      so any pair near the top is a fine choice; prefer the simpler tree (smaller depth, bigger leaves) on a tie.
df_scores.sort_values('auc', ascending=False).head()

In [ ]:
# GOAL: the final tree of this lesson.
dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=15)
dt.fit(X_train, y_train)

In [ ]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

## 6.6 Ensembles and random forest

- board of experts
- ensembling models
- random forest: ensembling decision trees
- tuning random forest

Many trees, each trained on a random subset of rows (bootstrap) and of features per split, then their
probabilities are **averaged**. The randomness is the point: identical trees would make identical mistakes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# GOAL: validation AUC as the number of trees grows from 10 to 200.
# WHY: random_state=1 fixes the randomness, so reruns give the same forest.
# PYTHON: range(10, 201, 10) = 10, 20, ..., 200 (the stop value 201 is excluded).
#         n_jobs=-1 trains the trees in parallel on all CPU cores (added here; same result, faster).
scores = []

for n in range(10, 201, 10):
    rf = RandomForestClassifier(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)

    scores.append((n, auc))

In [ ]:
df_scores = pd.DataFrame(scores, columns=['n_estimators', 'auc'])

In [ ]:
# WHY: AUC climbs fast, then goes flat: more trees never hurt quality, they only cost time.
#      So n_estimators = "as many as you can afford"; tune the other parameters first.
plt.plot(df_scores.n_estimators, df_scores.auc)

In [ ]:
# GOAL: the same curve for three depths of the individual trees.
scores = []

for d in [5, 10, 15]:
    for n in range(10, 201, 10):
        rf = RandomForestClassifier(n_estimators=n,
                                    max_depth=d,
                                    random_state=1,
                                    n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_pred)

        scores.append((d, n, auc))

In [ ]:
columns = ['max_depth', 'n_estimators', 'auc']
df_scores = pd.DataFrame(scores, columns=columns)

In [ ]:
# PYTHON: one plt.plot per depth draws several lines on the same axes; label= feeds plt.legend().
for d in [5, 10, 15]:
    df_subset = df_scores[df_scores.max_depth == d]

    plt.plot(df_subset.n_estimators, df_subset.auc,
             label='max_depth=%d' % d)

plt.legend()

In [ ]:
# GOAL: the plot as numbers: mean and best AUC per depth over all forest sizes.
# PYTHON: groupby('max_depth').auc.agg([...]) -> one row per depth with the listed statistics.
df_scores.groupby('max_depth').auc.agg(['mean', 'max']).round(3)

In [ ]:
# The lecture's pick from the plot.
max_depth = 10

In [ ]:
# GOAL: with max_depth fixed, tune the leaf size.
scores = []

for s in [1, 3, 5, 10, 50]:
    for n in range(10, 201, 10):
        rf = RandomForestClassifier(n_estimators=n,
                                    max_depth=max_depth,
                                    min_samples_leaf=s,
                                    random_state=1,
                                    n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_pred)

        scores.append((s, n, auc))

In [ ]:
columns = ['min_samples_leaf', 'n_estimators', 'auc']
df_scores = pd.DataFrame(scores, columns=columns)

In [ ]:
# PYTHON: zip(values, colors) walks two lists in step: (1, 'black'), (3, 'blue'), ...
colors = ['black', 'blue', 'orange', 'red', 'grey']
values = [1, 3, 5, 10, 50]

for s, col in zip(values, colors):
    df_subset = df_scores[df_scores.min_samples_leaf == s]

    plt.plot(df_subset.n_estimators, df_subset.auc,
             color=col,
             label='min_samples_leaf=%d' % s)

plt.legend()

In [ ]:
df_scores.groupby('min_samples_leaf').auc.agg(['mean', 'max']).round(3)

In [ ]:
# The lecture's pick from the plot (1, 3 and 5 are close; 50 is clearly worse).
min_samples_leaf = 3

In [ ]:
# GOAL: the final forest of this lesson: 200 trees, max_depth=10, min_samples_leaf=3.
rf = RandomForestClassifier(n_estimators=200,
                            max_depth=max_depth,
                            min_samples_leaf=min_samples_leaf,
                            random_state=1,
                            n_jobs=-1)
rf.fit(X_train, y_train)

Other useful parameters:

* `max_features`: how many features each split may look at (the source of the "random feature subsets")
* `bootstrap`: whether each tree sees a resampled copy of the rows

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

## 6.7 Gradient boosting and XGBoost

- gradient boosting vs random forest
- installing XGBoost
- training the first model
- performance monitoring
- parsing xgboost's monitoring output

Random forest trains its trees **independently** and averages them. Boosting trains them **in sequence**:
each new tree is fitted to the errors the previous ones still make, and its correction is scaled by `eta`.

In [ ]:
# PYTHON: install once in your environment (pip install xgboost), not from inside the notebook.
import xgboost as xgb

In [ ]:
# GOAL: wrap the matrices in XGBoost's own data structure, DMatrix (features + labels, optimised for training).
# WHY: feature_names makes the trees and the importances readable ('records=no' instead of 'f17').
# PYTHON: XGBoost 3 wants a LIST of strings; get_feature_names_out() returns a NumPy array, hence list(...).
features_train = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features_train)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features_train)

In [ ]:
# GOAL: a first model, 10 boosting rounds (= 10 trees).
# WHY: eta = learning rate (how much of each tree's correction is kept), max_depth = depth of each tree,
#      min_child_weight ~ minimal leaf size (like min_samples_leaf). 'binary:logistic' -> outputs probabilities.
#      seed fixes the randomness; nthread = CPU threads.
# PYTHON: the parameters are a plain dict; xgb.train(params, data, num_boost_round) returns the trained booster.
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'binary:logistic',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=10)

In [ ]:
# PYTHON: a booster's predict takes a DMatrix (not a NumPy array) and returns P(default) directly.
y_pred = model.predict(dval)
roc_auc_score(y_val, y_pred)

In [ ]:
# GOAL: a WATCHLIST: datasets XGBoost scores after every round, so we can watch it learn.
# PYTHON: a list of (DMatrix, name) tuples; the names appear in the printout ('train-auc', 'val-auc').
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [ ]:
%%capture output
# GOAL: 200 rounds, AUC printed every 5 rounds for train and val. %%capture stores the printout in 'output'.
# PYTHON: %%capture must be the FIRST line of the cell; nothing is shown, everything printed goes into output.stdout.
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'binary:logistic',
    'eval_metric': 'auc',

    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=200,
                  verbose_eval=5,
                  evals=watchlist)

In [ ]:
s = output.stdout
print(s[:200])

In [ ]:
# GOAL: text -> DataFrame. Each line looks like '[5]\ttrain-auc:0.93\tval-auc:0.81'.
# PYTHON: split('\t') cuts at tabs; strip('[]') removes the brackets; split(':')[1] takes the number part.
def parse_xgb_output(output):
    results = []

    for line in output.stdout.strip().split('\n'):
        it_line, train_line, val_line = line.split('\t')

        it = int(it_line.strip('[]'))
        train = float(train_line.split(':')[1])
        val = float(val_line.split(':')[1])

        results.append((it, train, val))

    columns = ['num_iter', 'train_auc', 'val_auc']
    df_results = pd.DataFrame(results, columns=columns)
    return df_results

In [ ]:
df_score = parse_xgb_output(output)

In [ ]:
# WHY: train AUC runs towards 1.0 while validation flattens and then slowly drops: OVERFITTING again.
#      Boosting keeps fitting the training errors, so the number of rounds is itself a parameter to tune.
plt.plot(df_score.num_iter, df_score.train_auc, label='train')
plt.plot(df_score.num_iter, df_score.val_auc, label='val')
plt.legend()

In [ ]:
plt.plot(df_score.num_iter, df_score.val_auc, label='val')
plt.legend()

In [ ]:
# GOAL: the production way to get the same curve: ask xgb.train to fill a dictionary (no text parsing).
# WHY: parsing printed text breaks the day XGBoost prints one extra line (a warning, a format change).
#      evals_result is structured data: evals_result['val']['auc'] = the AUC after every round.
# PYTHON: evals_result={} is filled IN PLACE by xgb.train; verbose_eval=False prints nothing.
evals_result = {}
model = xgb.train(xgb_params, dtrain, num_boost_round=200,
                  evals=watchlist, evals_result=evals_result, verbose_eval=False)

df_score2 = pd.DataFrame({
    'num_iter': range(200),
    'train_auc': evals_result['train']['auc'],
    'val_auc': evals_result['val']['auc'],
})

# Same numbers as the parsed text (which only kept every 5th round), to the printed 5 decimals.
df_score2.iloc[::5].head()

## 6.8 XGBoost parameter tuning

Tuning the following parameters, in this order, one at a time:

* `eta`
* `max_depth`
* `min_child_weight`

The lecture repeats the `%%capture` cell by hand for every value. Here one helper does a run and returns the
validation curve, so every value the lesson compares ends up on the same plot.

In [ ]:
# GOAL: one function = one tuning run: params in, (model, learning-curve DataFrame) out.
# PYTHON: {**base, **changes} builds a NEW dict: base's keys, then the changed keys overwrite them.
base_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'binary:logistic',
    'eval_metric': 'auc',

    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}


def train_xgb(params, num_boost_round=200):
    evals_result = {}
    model = xgb.train(params, dtrain, num_boost_round=num_boost_round,
                      evals=watchlist, evals_result=evals_result, verbose_eval=False)

    df_curve = pd.DataFrame({
        'num_iter': range(num_boost_round),
        'train_auc': evals_result['train']['auc'],
        'val_auc': evals_result['val']['auc'],
    })
    return model, df_curve


def plot_curves(scores, ylim):
    for key, df_curve in scores.items():
        plt.plot(df_curve.num_iter, df_curve.val_auc, label=key)
    plt.ylim(*ylim)
    plt.legend()


def best_rounds(scores):
    # PYTHON: idxmax() = the row label of the maximum -> the round with the best validation AUC.
    rows = [(key, df.val_auc.max(), df.val_auc.idxmax()) for key, df in scores.items()]
    return pd.DataFrame(rows, columns=['run', 'best_val_auc', 'at_round']).round(4)

In [ ]:
# GOAL: tune eta (the learning rate) first: 0.3 (the value so far), 1.0, 0.1, 0.05, 0.01.
# WHY: big eta learns fast and overfits early (1.0 falls below the zoom window); 0.01 is still climbing
#      at round 200. The lecture takes eta=0.1: near the top within ~70 rounds and stable after it.
#      On XGBoost 3 eta=0.05 ends a few thousandths higher, but only at round ~190 and still rising
#      (it wants more rounds). Differences that small are within split-to-split noise; we keep 0.1.
scores = {}

for eta in [0.3, 1.0, 0.1, 0.05, 0.01]:
    params = {**base_params, 'eta': eta}
    model, scores['eta=%s' % eta] = train_xgb(params)

plot_curves(scores, ylim=(0.8, 0.84))
best_rounds(scores)

In [ ]:
# GOAL: with eta=0.1 fixed, tune max_depth: 6 (baseline), 3, 4, 10.
scores = {}

for depth in [6, 3, 4, 10]:
    params = {**base_params, 'eta': 0.1, 'max_depth': depth}
    model, scores['max_depth=%s' % depth] = train_xgb(params)

best_rounds(scores)

In [ ]:
# WHY: max_depth=10 is clearly worst, so the lecture drops it to zoom in on the others. Shallow trees win:
#      in boosting each tree only corrects the remaining error, so it does not need to be deep.
del scores['max_depth=10']

plot_curves(scores, ylim=(0.8, 0.84))

In [ ]:
# GOAL: with eta=0.1 and max_depth=3, tune min_child_weight: 1 (baseline), 10, 30.
# WHY: the three best AUCs differ by about 0.001: noise level, the parameter barely matters here.
#      The lecture keeps the default 1.
scores = {}

for mcw in [1, 10, 30]:
    params = {**base_params, 'eta': 0.1, 'max_depth': 3, 'min_child_weight': mcw}
    model, scores['min_child_weight=%s' % mcw] = train_xgb(params)

plot_curves(scores, ylim=(0.82, 0.84))
best_rounds(scores)

In [ ]:
# GOAL: the tuned model: eta=0.1, max_depth=3, min_child_weight=1, 175 rounds.
# WHY: 175 is read from the curves: around there validation AUC is at its best and still flat.
xgb_params = {
    'eta': 0.1,
    'max_depth': 3,
    'min_child_weight': 1,

    'objective': 'binary:logistic',
    'eval_metric': 'auc',

    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=175)

Other parameters: https://xgboost.readthedocs.io/en/latest/parameter.html

Useful ones:

* `subsample` and `colsample_bytree`: share of rows / features each tree sees (random-forest-style diversity)
* `lambda` and `alpha`: L2 / L1 regularisation of the leaf weights

## 6.9 Selecting the final model

- choosing between xgboost, random forest and decision tree
- training the final model
- saving the model (Module 5's pickle recipe; not repeated here)

In [ ]:
# GOAL: the three tuned models on the SAME validation set: a fair comparison.
dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=15)
dt.fit(X_train, y_train)

y_pred = dt.predict_proba(X_val)[:, 1]
auc_dt = roc_auc_score(y_val, y_pred)
auc_dt

In [ ]:
rf = RandomForestClassifier(n_estimators=200,
                            max_depth=10,
                            min_samples_leaf=3,
                            random_state=1,
                            n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict_proba(X_val)[:, 1]
auc_rf = roc_auc_score(y_val, y_pred)
auc_rf

In [ ]:
model = xgb.train(xgb_params, dtrain, num_boost_round=175)

y_pred = model.predict(dval)
auc_xgb = roc_auc_score(y_val, y_pred)
auc_xgb

In [ ]:
# WHY: typical result on tabular data: boosting > forest > single tree. The price: XGBoost has more knobs
#      and overfits faster, so it needs the careful tuning of 6.8; a random forest works well almost out of the box.
pd.Series({'decision tree': auc_dt, 'random forest': auc_rf, 'xgboost': auc_xgb}).round(3)

In [ ]:
# GOAL: the final recipe: retrain the winner on train + validation (df_full_train), score ONCE on test.
# PYTHON: df_full_train still has its shuffled index and the 'status' column; renumber, take y, drop status.
df_full_train = df_full_train.reset_index(drop=True)
y_full_train = (df_full_train.status == 'default').astype(int).values
del df_full_train['status']

In [ ]:
# WHY: a NEW DictVectorizer, fitted on df_full_train only; the test set is only transformed.
#      .fillna(0) as in every model before: the lecture skips it here, so its final model would see NaN
#      where all the validated models saw 0 (XGBoost accepts NaN, but it is no longer the validated pipeline).
dicts_full_train = df_full_train.fillna(0).to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)

dicts_test = df_test.fillna(0).to_dict(orient='records')
X_test = dv.transform(dicts_test)

In [ ]:
# PYTHON: list(...) again: XGBoost 3 rejects the NumPy array the lecture passes here (TypeError).
#         The test DMatrix needs no label: we only predict on it.
features_full = list(dv.get_feature_names_out())

dfulltrain = xgb.DMatrix(X_full_train, label=y_full_train, feature_names=features_full)
dtest = xgb.DMatrix(X_test, feature_names=features_full)

In [ ]:
model = xgb.train(xgb_params, dfulltrain, num_boost_round=175)

In [ ]:
# WHY: test AUC close to the validation AUC above = the tuning did not overfit the validation set and the
#      model generalises. A big drop would mean we tuned to the noise of one validation split.
y_pred = model.predict(dtest)
auc_test = roc_auc_score(y_test, y_pred)
auc_test, auc_xgb

## 6.10 Summary

| Model | How it predicts | Main knobs | Watch out |
|---|---|---|---|
| Decision tree | a chain of if/else splits learned from data | `max_depth`, `min_samples_leaf` | unlimited depth memorises the training set (train AUC 1.0) |
| Split finding | try every threshold of every feature, keep the lowest impurity | criterion: Gini / entropy (MSE for regression) | needs NaN filled (`fillna(0)`) in scikit-learn |
| Random forest | average of many randomised trees, trained independently | `max_depth`, `min_samples_leaf`, `n_estimators`, `max_features` | more trees never hurt, they only cost time |
| Gradient boosting (XGBoost) | trees in sequence, each corrects the previous errors | `eta`, `max_depth`, `min_child_weight`, number of rounds | overfits with rounds: watch the validation curve |
| Model selection | compare tuned models on one validation set, retrain the winner on train + val, test once | — | keep the final pipeline identical to the validated one |

## 6.11 Explore more

- No EDA or feature engineering was done on this dataset: try it.
- Random forest: tune `max_features` and `bootstrap`. Try `ExtraTreesClassifier` (random thresholds instead of the best one).
- XGBoost handles NaN natively: skip `fillna(0)` and compare.
- XGBoost: try `subsample` and `colsample_bytree`.
- Feature importance: `rf.feature_importances_` and `model.get_score(importance_type='gain')` (cell below).
- Trees for regression: `DecisionTreeRegressor`, `RandomForestRegressor`, `objective='reg:squarederror'` (that is HW6).

In [ ]:
# GOAL: which features the tuned forest relies on most (total impurity reduction they bring, summed over all splits).
# WHY: one-hot encoding spreads a categorical feature over several columns ('home=owner', 'home=rent', ...),
#      so its importance is split too: add them up before comparing it with a numerical feature.
# PYTHON: pd.Series(values, index=names) pairs each importance with its column name; sort_values ranks them.
# features_train = the column names of X_train (6.7); the forest rf from 6.9 was trained on X_train.
pd.Series(rf.feature_importances_, index=features_train).sort_values(ascending=False).head(10).round(3)